# Job Market Skill Gap Analysis - Exploratory Data Analysis

This notebook demonstrates the complete pipeline and provides exploratory analysis of job market data.

## 1. Setup and Imports

In [ ]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from collections import Counter

# Project modules
from scraping.scraper import JobScraper
from nlp.skill_extractor import SkillExtractor
from backend.database import FileStorage

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("✓ All imports successful")

## 2. Data Collection

In [ ]:
# Initialize scraper
scraper = JobScraper()

# Collect jobs
jobs_df = scraper.scrape_jobs(
    keywords=["data scientist", "data engineer", "machine learning engineer"],
    locations=["Remote", "United States"],
    max_jobs=200
)

# Save to file
scraper.save_jobs(jobs_df)

print(f"Collected {len(jobs_df)} job postings")
jobs_df.head()

## 3. Job Market Overview

In [ ]:
# Basic statistics
print("Job Market Summary")
print("=" * 50)
print(f"Total Jobs: {len(jobs_df)}")
print(f"Unique Companies: {jobs_df['company'].nunique()}")
print(f"Unique Titles: {jobs_df['title'].nunique()}")
print(f"Unique Locations: {jobs_df['location'].nunique()}")
print("\nTop 10 Companies:")
print(jobs_df['company'].value_counts().head(10))

In [ ]:
# Job title distribution
title_counts = jobs_df['title'].value_counts().head(10)

fig = px.bar(
    x=title_counts.values,
    y=title_counts.index,
    orientation='h',
    title='Top 10 Job Titles',
    labels={'x': 'Count', 'y': 'Job Title'}
)
fig.update_layout(height=400, yaxis={'categoryorder': 'total ascending'})
fig.show()

In [ ]:
# Location distribution
location_counts = jobs_df['location'].value_counts().head(10)

fig = px.pie(
    values=location_counts.values,
    names=location_counts.index,
    title='Job Locations'
)
fig.show()

## 4. NLP Skill Extraction

In [ ]:
# Initialize skill extractor
extractor = SkillExtractor()

# Extract skills from all jobs
jobs_with_skills = extractor.extract_skills_from_jobs(jobs_df)

print(f"Extracted skills from {len(jobs_with_skills)} jobs")
print(f"Average skills per job: {jobs_with_skills['skill_count'].mean():.1f}")
print(f"Median skills per job: {jobs_with_skills['skill_count'].median():.0f}")

jobs_with_skills.head()

In [ ]:
# Distribution of skills per job
fig = px.histogram(
    jobs_with_skills,
    x='skill_count',
    title='Distribution of Skills per Job Posting',
    labels={'skill_count': 'Number of Skills', 'count': 'Frequency'}
)
fig.show()

## 5. Skill Demand Analysis

In [ ]:
# Compute skill demand
demand_df = extractor.compute_skill_demand(jobs_with_skills)

print(f"Found {len(demand_df)} unique skills")
print("\nTop 20 Most Demanded Skills:")
print(demand_df.head(20))

In [ ]:
# Visualize top skills
top_20 = demand_df.head(20)

fig = px.bar(
    top_20,
    x='percentage',
    y='skill',
    orientation='h',
    title='Top 20 Skills by Demand',
    labels={'percentage': 'Percentage of Jobs (%)', 'skill': 'Skill'},
    color='demand_level',
    color_discrete_map={'High': '#d32f2f', 'Medium': '#f57c00', 'Low': '#388e3c'}
)
fig.update_layout(height=600, yaxis={'categoryorder': 'total ascending'})
fig.show()

In [ ]:
# Demand level distribution
demand_level_counts = demand_df['demand_level'].value_counts()

fig = px.pie(
    values=demand_level_counts.values,
    names=demand_level_counts.index,
    title='Skill Demand Levels',
    color=demand_level_counts.index,
    color_discrete_map={'High': '#d32f2f', 'Medium': '#f57c00', 'Low': '#388e3c'}
)
fig.show()

## 6. Category Analysis

In [ ]:
# Skills by category
if 'category' in demand_df.columns:
    category_stats = demand_df.groupby('category').agg({
        'job_count': 'sum',
        'skill': 'count'
    }).reset_index()
    category_stats.columns = ['category', 'total_mentions', 'unique_skills']
    category_stats = category_stats.sort_values('total_mentions', ascending=False)
    
    print("Skills by Category:")
    print(category_stats)
    
    # Visualize
    fig = px.bar(
        category_stats,
        x='category',
        y='total_mentions',
        title='Total Skill Mentions by Category',
        labels={'total_mentions': 'Total Mentions', 'category': 'Category'}
    )
    fig.update_layout(xaxis_tickangle=-45)
    fig.show()

## 7. Sample Resume Analysis

In [ ]:
# Sample resume
sample_resume = """
John Doe
Data Scientist

SKILLS
- Programming: Python, SQL, R
- Machine Learning: scikit-learn, TensorFlow
- Data Analysis: Pandas, NumPy
- Visualization: Matplotlib, Tableau
- Tools: Git, Jupyter

EXPERIENCE
Data Analyst at Tech Company (2020-2023)
- Built predictive models using Python and scikit-learn
- Analyzed large datasets with SQL and Pandas
- Created dashboards in Tableau
"""

# Extract skills
resume_data = extractor.extract_from_resume(sample_resume)

print(f"Extracted {resume_data['skill_count']} skills from resume:")
print(resume_data['skills'])

## 8. Skill Gap Analysis

In [ ]:
# Compare resume skills with market demand
gap_df = extractor.compare_skills(
    student_skills=resume_data['skills'],
    market_demand=demand_df,
    top_n=30
)

print("\nSkill Gap Analysis:")
print(gap_df)

In [ ]:
# Summary statistics
matched = gap_df['student_has'].sum()
missing = (~gap_df['student_has']).sum()
missing_high = gap_df[(gap_df['gap']) & (gap_df['demand_level'] == 'High')].shape[0]
missing_medium = gap_df[(gap_df['gap']) & (gap_df['demand_level'] == 'Medium')].shape[0]

print(f"\nSkill Gap Summary:")
print(f"Skills Matched: {matched}")
print(f"Skills Missing: {missing}")
print(f"Missing High Demand: {missing_high}")
print(f"Missing Medium Demand: {missing_medium}")

In [ ]:
# Visualize gap analysis
fig = go.Figure()

# Add bars for market demand
fig.add_trace(go.Bar(
    y=gap_df['skill'],
    x=gap_df['percentage'],
    name='Market Demand',
    orientation='h',
    marker_color='lightblue'
))

# Add markers for student skills
student_skills_df = gap_df[gap_df['student_has'] == True]
fig.add_trace(go.Scatter(
    y=student_skills_df['skill'],
    x=student_skills_df['percentage'],
    mode='markers',
    name='Your Skills',
    marker=dict(size=12, color='green', symbol='diamond')
))

fig.update_layout(
    title='Skill Gap Analysis: Market vs Your Skills',
    xaxis_title='Percentage of Jobs (%)',
    yaxis_title='Skill',
    height=700,
    yaxis={'categoryorder': 'total ascending'}
)
fig.show()

## 9. Key Recommendations

In [ ]:
# Missing high-demand skills
missing_high_demand = gap_df[
    (gap_df['gap']) & (gap_df['demand_level'] == 'High')
].sort_values('percentage', ascending=False)

print("\n🎯 TOP PRIORITY: Missing High-Demand Skills")
print("=" * 50)
for _, row in missing_high_demand.head(5).iterrows():
    print(f"• {row['skill']}: {row['percentage']:.1f}% of jobs require this")

# Missing medium-demand skills
missing_medium_demand = gap_df[
    (gap_df['gap']) & (gap_df['demand_level'] == 'Medium')
].sort_values('percentage', ascending=False)

print("\n⚠️ MEDIUM PRIORITY: Missing Medium-Demand Skills")
print("=" * 50)
for _, row in missing_medium_demand.head(5).iterrows():
    print(f"• {row['skill']}: {row['percentage']:.1f}% of jobs require this")

# Matched high-value skills
matched_high = gap_df[
    (gap_df['student_has']) & (gap_df['demand_level'] == 'High')
]

print("\n✅ STRENGTHS: Your High-Demand Skills")
print("=" * 50)
for _, row in matched_high.iterrows():
    print(f"• {row['skill']}: {row['percentage']:.1f}% of jobs")

## 10. Save Results

In [ ]:
# Save processed data
jobs_with_skills.to_csv('../data/processed/jobs_with_skills.csv', index=False)
demand_df.to_csv('../data/processed/skill_demand.csv', index=False)
gap_df.to_csv('../data/processed/skill_gap_analysis.csv', index=False)

print("✓ Results saved to data/processed/")